In [1]:
# %% 셀 1: 데이터 로드 + 임베딩 로드 + STT 로드 + train/eval 분리
import os, json, random
import numpy as np
import torch
import polars as pl
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor, as_completed

POS_DIR = "./data/8_telop_position"
STT_DIR = "./data/4_stt_results"
EMB_PATH = "./data/8_text_embeddings.pt"
GRID_W = 80
GRID_H = 80
CELL_SIZE_X = 9
CELL_SIZE_Y = 16
EVAL_PER_CHANNEL = 5
SEED = 42
NUM_WORKERS = 32
FPS = 10

# ── 임베딩 로드 ──
text2emb = torch.load(EMB_PATH, weights_only=True)
EMB_DIM = next(iter(text2emb.values())).shape[0]
ZERO_EMB = torch.zeros(EMB_DIM)
print(f"✅ 임베딩 로드: {len(text2emb):,}개  |  dim={EMB_DIM}")


def stt_frames_to_segments(df_stt):
    rows = df_stt.sort("frame_num").to_dicts()
    segments = []
    cur_text = None
    cur_start_frame = None
    prev_frame = None

    for r in rows:
        t = r["stt_text"]
        f = int(r["frame_num"])

        if t != cur_text:
            if cur_text is not None and cur_text != "":
                segments.append({
                    "start": cur_start_frame / FPS,
                    "end": (prev_frame + 1) / FPS,
                    "text": cur_text.strip(),
                })
            cur_text = t
            cur_start_frame = f
        prev_frame = f

    if cur_text is not None and cur_text != "":
        segments.append({
            "start": cur_start_frame / FPS,
            "end": (prev_frame + 1) / FPS,
            "text": cur_text.strip(),
        })
    return segments


def load_one(args):
    channel, path = args
    with open(path, "r") as f:
        data = json.load(f)

    instances = data.get("instances", [])
    duration = data.get("duration", 0.1)
    if instances:
        duration = max(duration, max(inst["end_sec"] for inst in instances))

    video_name = data.get("video", "")

    inst_list = []
    for inst in instances:
        gx = int(np.clip(round(inst["grid_x"]), 0, GRID_W - 1))
        gy = int(np.clip(round(inst["grid_y"]), 0, GRID_H - 1))
        gw = int(np.clip(round(inst["grid_w"]), 1, GRID_W))
        gh = int(np.clip(round(inst["grid_h"]), 1, GRID_H))

        inst_list.append({
            "text": inst["text"],
            "text_len": len(inst["text"]),
            "start": inst["start_sec"],
            "end": inst["end_sec"],
            "x": gx, "y": gy, "w": gw, "h": gh,
        })

    # STT 로드
    fname = os.path.basename(path)[:-5]
    stt_path = os.path.join(STT_DIR, channel, fname + ".parquet")
    stt_segments = []
    if os.path.exists(stt_path):
        try:
            df_stt = pl.read_parquet(stt_path, glob=False)
            stt_segments = stt_frames_to_segments(df_stt)
        except:
            pass

    return {
        "channel": channel,
        "video_name": video_name,
        "instances": inst_list,
        "stt_segments": stt_segments,
        "duration": duration,
    }


# ── 파일 목록 수집 ──
json_paths = []
for channel in sorted(os.listdir(POS_DIR)):
    ch_dir = os.path.join(POS_DIR, channel)
    if not os.path.isdir(ch_dir):
        continue
    for fname in sorted(os.listdir(ch_dir)):
        if fname.endswith(".json"):
            json_paths.append((channel, os.path.join(ch_dir, fname)))

print(f"📁 파일 수: {len(json_paths):,}개")

# ── 병렬 로드 ──
channel_set = set()
samples = []

with ProcessPoolExecutor(max_workers=NUM_WORKERS) as pool:
    futures = {pool.submit(load_one, args): args for args in json_paths}
    for fut in tqdm(as_completed(futures), total=len(futures), desc="로드"):
        result = fut.result()
        channel_set.add(result["channel"])
        samples.append(result)

channels = sorted(channel_set)
channel2id = {ch: i for i, ch in enumerate(channels)}

# ── 통계 ──
n_with_inst = sum(1 for s in samples if len(s["instances"]) > 0)
n_without_inst = sum(1 for s in samples if len(s["instances"]) == 0)
stt_counts = np.array([len(s["stt_segments"]) for s in samples])
inst_counts = np.array([len(s["instances"]) for s in samples])

print(f"\n📊 영상별 STT 세그먼트 수")
print(f"  mean: {stt_counts.mean():.1f}  median: {np.median(stt_counts):.0f}  "
      f"p99: {np.percentile(stt_counts, 99):.0f}  max: {stt_counts.max()}")

# ── train/eval 분리 ──
rng = random.Random(SEED)
by_channel = {}
for s in samples:
    if s["channel"] not in by_channel:
        by_channel[s["channel"]] = []
    by_channel[s["channel"]].append(s)

train_samples, eval_samples = [], []
for ch, ch_samples in by_channel.items():
    rng.shuffle(ch_samples)
    n_eval = min(EVAL_PER_CHANNEL, len(ch_samples))
    eval_samples.extend(ch_samples[:n_eval])
    train_samples.extend(ch_samples[n_eval:])

print(f"\n✅ 영상: {len(samples):,}개 (텔롭 있음: {n_with_inst:,}  없음: {n_without_inst:,})  |  채널: {len(channels)}개")
print(f"✅ train: {len(train_samples):,}  |  eval: {len(eval_samples):,}")
print(f"📊 인스턴스 수 — mean: {inst_counts.mean():.0f}  p99: {np.percentile(inst_counts, 99):.0f}  max: {inst_counts.max()}")

✅ 임베딩 로드: 2,167,019개  |  dim=1024
📁 파일 수: 66,400개


로드: 100%|██████████| 66400/66400 [00:10<00:00, 6633.21it/s]



📊 영상별 STT 세그먼트 수
  mean: 12.8  median: 10  p99: 51  max: 151

✅ 영상: 66,400개 (텔롭 있음: 59,876  없음: 6,524)  |  채널: 664개
✅ train: 63,080  |  eval: 3,320
📊 인스턴스 수 — mean: 56  p99: 403  max: 4251


In [2]:
# %% 셀 2: Dataset + DataLoader
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

BATCH_SIZE = 2
NUM_WORKERS = 8
BIN_SIZE = 0.1
MAX_T = 2000
K_TELOP = 128
TELOP_FEAT_DIM = 5


class FrameMaskDataset(Dataset):
    def __init__(self, samples, channel2id, text2emb):
        self.samples = samples
        self.channel2id = channel2id
        self.text2emb = text2emb

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        duration = max(s["duration"], 0.1)
        T = min(int(np.ceil(duration / BIN_SIZE)), MAX_T)

        channel_id = self.channel2id.get(s["channel"], 0)
        channel_emb = self.text2emb.get(s["channel"], ZERO_EMB)
        video_emb = self.text2emb.get(s["video_name"], ZERO_EMB)

        instances = s["instances"]
        n_inst = len(instances)

        if n_inst > 0:
            inst_starts = np.array([inst["start"] for inst in instances])
            inst_ends = np.array([inst["end"] for inst in instances])
            inst_tlens = np.array([inst["text_len"] for inst in instances])
            inst_cx = np.array([inst["x"] for inst in instances], dtype=np.float32)
            inst_cy = np.array([inst["y"] for inst in instances], dtype=np.float32)
            inst_w = np.array([inst["w"] for inst in instances], dtype=np.float32)
            inst_h = np.array([inst["h"] for inst in instances], dtype=np.float32)

            inst_embs = torch.stack([
                self.text2emb.get(inst["text"], ZERO_EMB) for inst in instances
            ])

            ch_sim = F.cosine_similarity(inst_embs, channel_emb.unsqueeze(0), dim=-1).numpy()
            vid_sim = F.cosine_similarity(inst_embs, video_emb.unsqueeze(0), dim=-1).numpy()
        else:
            inst_starts = np.array([])
            inst_ends = np.array([])
            inst_tlens = np.array([])
            inst_cx = np.array([], dtype=np.float32)
            inst_cy = np.array([], dtype=np.float32)
            inst_w = np.array([], dtype=np.float32)
            inst_h = np.array([], dtype=np.float32)
            inst_embs = torch.zeros(0, EMB_DIM)
            ch_sim = np.array([])
            vid_sim = np.array([])

        stt_segments = s["stt_segments"]
        n_stt = len(stt_segments)
        if n_stt > 0:
            stt_starts = np.array([seg["start"] for seg in stt_segments])
            stt_ends = np.array([seg["end"] for seg in stt_segments])
            stt_embs = torch.stack([
                self.text2emb.get(seg["text"], ZERO_EMB) for seg in stt_segments
            ])
        else:
            stt_starts = np.array([])
            stt_ends = np.array([])
            stt_embs = torch.zeros(0, EMB_DIM)

        # ── 프레임별 ──
        time_feats = np.zeros((T, 1), dtype=np.float32)  # 시간 위치 (spatial query용)
        telop_feats = np.zeros((T, K_TELOP, TELOP_FEAT_DIM), dtype=np.float32)
        telop_mask = np.zeros((T, K_TELOP), dtype=np.bool_)
        gt_masks = np.zeros((T, GRID_H, GRID_W), dtype=np.float32)

        for t in range(T):
            t_sec = t * BIN_SIZE

            time_feats[t, 0] = t / max(T - 1, 1)

            if n_inst > 0:
                active = (inst_starts <= t_sec) & (inst_ends > t_sec)
                active_idx = np.where(active)[0]
            else:
                active_idx = np.array([], dtype=np.int64)

            # 활성 STT 찾기
            stt_active_emb = None
            if n_stt > 0:
                stt_active = (stt_starts <= t_sec) & (stt_ends > t_sec)
                stt_active_idx = np.where(stt_active)[0]
                if len(stt_active_idx) > 0:
                    stt_active_emb = stt_embs[stt_active_idx[0]]

            for j, ai in enumerate(active_idx[:K_TELOP]):
                telop_feats[t, j, 0] = inst_tlens[ai] / 200.0
                telop_feats[t, j, 2] = ch_sim[ai]
                telop_feats[t, j, 3] = vid_sim[ai]

                if stt_active_emb is not None:
                    telop_feats[t, j, 1] = F.cosine_similarity(
                        inst_embs[ai].unsqueeze(0),
                        stt_active_emb.unsqueeze(0)
                    ).item()
                    telop_feats[t, j, 4] = 1.0

                telop_mask[t, j] = True

            # GT mask
            for ai in active_idx:
                cx, cy = inst_cx[ai], inst_cy[ai]
                w, h = inst_w[ai], inst_h[ai]
                x0 = int(np.clip(round(cx - w / 2), 0, GRID_W - 1))
                y0 = int(np.clip(round(cy - h / 2), 0, GRID_H - 1))
                x1 = int(np.clip(round(cx + w / 2), 1, GRID_W))
                y1 = int(np.clip(round(cy + h / 2), 1, GRID_H))
                gt_masks[t, y0:y1, x0:x1] = 1.0

        return {
            "channel_id": torch.tensor(channel_id, dtype=torch.long),
            "time_feats": torch.from_numpy(time_feats),
            "telop_feats": torch.from_numpy(telop_feats),
            "telop_mask": torch.from_numpy(telop_mask),
            "gt_masks": torch.from_numpy(gt_masks),
            "T": T,
        }


def collate_fn(batch):
    max_T = max(b["T"] for b in batch)
    B = len(batch)

    channel_ids = torch.zeros(B, dtype=torch.long)
    time_feats = torch.zeros(B, max_T, 1)
    telop_feats = torch.zeros(B, max_T, K_TELOP, TELOP_FEAT_DIM)
    telop_mask = torch.zeros(B, max_T, K_TELOP, dtype=torch.bool)
    gt_masks = torch.zeros(B, max_T, GRID_H, GRID_W)
    time_mask = torch.zeros(B, max_T, dtype=torch.bool)

    for i, b in enumerate(batch):
        T = b["T"]
        channel_ids[i] = b["channel_id"]
        time_feats[i, :T] = b["time_feats"]
        telop_feats[i, :T] = b["telop_feats"]
        telop_mask[i, :T] = b["telop_mask"]
        gt_masks[i, :T] = b["gt_masks"]
        time_mask[i, :T] = True

    return {
        "channel_id": channel_ids,
        "time_feats": time_feats,
        "telop_feats": telop_feats,
        "telop_mask": telop_mask,
        "gt_masks": gt_masks,
        "time_mask": time_mask,
    }


train_ds = FrameMaskDataset(train_samples, channel2id, text2emb)
eval_ds = FrameMaskDataset(eval_samples, channel2id, text2emb)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)
eval_loader = DataLoader(eval_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True, collate_fn=collate_fn)

batch = next(iter(train_loader))
print(f"✅ 배치 확인")
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k}: {v.shape}")

pos_ratio = batch['gt_masks'][batch['time_mask']].mean().item()
print(f"\n  positive 비율 (배치 근사): {pos_ratio:.4f}")

✅ 배치 확인
  channel_id: torch.Size([2])
  time_feats: torch.Size([2, 587, 1])
  telop_feats: torch.Size([2, 587, 128, 5])
  telop_mask: torch.Size([2, 587, 128])
  gt_masks: torch.Size([2, 587, 80, 80])
  time_mask: torch.Size([2, 587])

  positive 비율 (배치 근사): 0.0426


In [4]:
# %% 셀 3: 모델 정의 (DETR 스타일, multi-token spatial memory, memory padding)
D_MODEL = 256
N_HEADS = 8
N_LAYERS_TEMPORAL = 4
N_LAYERS_SPATIAL = 4
D_FF = 512
DROPOUT = 0.1
POS_WEIGHT = 12.2
SPATIAL_H = 20
SPATIAL_W = 20
N_COND_TOKENS = 2
K_MEMORY = 8

INTRA_DIM = D_MODEL
INTRA_FEAT = 16


class IntraFrameModule(nn.Module):
    def __init__(self, d_out=INTRA_DIM, feat_dim=INTRA_FEAT, n_heads=4, k_memory=K_MEMORY):
        super().__init__()
        self.len_proj = nn.Sequential(nn.Linear(1, feat_dim), nn.GELU())
        self.stt_sim_proj = nn.Sequential(nn.Linear(1, feat_dim), nn.GELU())
        self.ch_sim_proj = nn.Sequential(nn.Linear(1, feat_dim), nn.GELU())
        self.vid_sim_proj = nn.Sequential(nn.Linear(1, feat_dim), nn.GELU())
        self.has_stt_proj = nn.Sequential(nn.Linear(1, feat_dim), nn.GELU())
        self.slot_proj = nn.Linear(feat_dim * 5, d_out)
        self.self_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)

        self.pool_query = nn.Parameter(torch.randn(1, 1, d_out))
        self.pool_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)

        self.memory_queries = nn.Parameter(torch.randn(1, k_memory, d_out))
        self.memory_attn = nn.MultiheadAttention(d_out, n_heads, batch_first=True, dropout=0.1)

    def forward(self, telop_feats, telop_mask):
        BT, K, _ = telop_feats.shape

        f_len = self.len_proj(telop_feats[:, :, 0:1])
        f_stt = self.stt_sim_proj(telop_feats[:, :, 1:2])
        f_ch = self.ch_sim_proj(telop_feats[:, :, 2:3])
        f_vid = self.vid_sim_proj(telop_feats[:, :, 3:4])
        f_has = self.has_stt_proj(telop_feats[:, :, 4:5])
        x = torch.cat([f_len, f_stt, f_ch, f_vid, f_has], dim=-1)
        x = self.slot_proj(x)

        any_active = telop_mask.any(dim=1)

        pad_mask = ~telop_mask
        all_padded = ~any_active
        pad_mask[all_padded] = False

        x_att, _ = self.self_attn(x, x, x, key_padding_mask=pad_mask)

        query = self.pool_query.expand(BT, -1, -1)
        pooled, _ = self.pool_attn(query, x_att, x_att, key_padding_mask=pad_mask)
        pooled = pooled.squeeze(1)
        pooled = pooled * any_active.unsqueeze(-1).float()

        mem_q = self.memory_queries.expand(BT, -1, -1)
        memory_tokens, _ = self.memory_attn(mem_q, x_att, x_att, key_padding_mask=pad_mask)
        memory_tokens = memory_tokens * any_active.unsqueeze(-1).unsqueeze(-1).float()

        return pooled, memory_tokens


class FrameMaskModel(nn.Module):
    def __init__(self, n_channels, d_model=D_MODEL,
                 n_heads=N_HEADS, d_ff=D_FF, dropout=DROPOUT):
        super().__init__()

        self.channel_id_emb = nn.Embedding(n_channels, d_model)

        self.intra_telop = IntraFrameModule(d_model, INTRA_FEAT)

        self.temporal_pos_proj = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )

        temporal_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.temporal_transformer = nn.TransformerEncoder(
            temporal_layer, num_layers=N_LAYERS_TEMPORAL
        )

        self.cond_type_emb = nn.Embedding(N_COND_TOKENS, d_model)

        self.spatial_queries = nn.Parameter(torch.randn(SPATIAL_H * SPATIAL_W, d_model))
        nn.init.normal_(self.spatial_queries, std=0.02)

        self.pos_x_emb = nn.Embedding(SPATIAL_H, d_model)
        self.pos_y_emb = nn.Embedding(SPATIAL_W, d_model)

        self.time_proj = nn.Sequential(
            nn.Linear(3, 64),
            nn.GELU(),
            nn.Linear(64, d_model),
        )

        spatial_decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True, activation="gelu",
        )
        self.spatial_decoder = nn.TransformerDecoder(
            spatial_decoder_layer, num_layers=N_LAYERS_SPATIAL
        )

        self.upsample = nn.Sequential(
            nn.ConvTranspose2d(d_model, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.ConvTranspose2d(64, 16, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(16),
            nn.GELU(),
            nn.Conv2d(16, 1, kernel_size=1),
        )

    def _get_xy_emb(self, device):
        pos_x = torch.arange(SPATIAL_H, device=device)
        pos_y = torch.arange(SPATIAL_W, device=device)
        grid_x, grid_y = torch.meshgrid(pos_x, pos_y, indexing='ij')
        return self.pos_x_emb(grid_x.flatten()) + self.pos_y_emb(grid_y.flatten())

    def forward(self, channel_id, time_feats, telop_feats, telop_mask, time_mask):
        B, T, _ = time_feats.shape
        device = time_feats.device

        ch_id = self.channel_id_emb(channel_id).unsqueeze(1).expand(-1, T, -1)

        telop_flat = telop_feats.reshape(B * T, K_TELOP, TELOP_FEAT_DIM)
        telop_mask_flat = telop_mask.reshape(B * T, K_TELOP)
        telop_pooled, telop_memory = self.intra_telop(telop_flat, telop_mask_flat)

        telop_pooled = telop_pooled.reshape(B, T, D_MODEL)

        # ── Temporal Transformer ──
        type_ids = torch.arange(N_COND_TOKENS, device=device)
        type_embs = self.cond_type_emb(type_ids)

        ch_id_temporal = ch_id + type_embs[0]
        telop_temporal = telop_pooled + type_embs[1]

        cond_tokens = torch.stack([ch_id_temporal, telop_temporal], dim=2)
        cond_tokens = cond_tokens.reshape(B, T * N_COND_TOKENS, D_MODEL)

        t_raw = time_feats[:, :, 0:1]
        t_sin = torch.sin(2 * torch.pi * t_raw)
        t_cos = torch.cos(2 * torch.pi * t_raw)
        t_input = torch.cat([t_raw, t_sin, t_cos], dim=-1)

        t_pos = self.temporal_pos_proj(t_input)
        t_pos = t_pos.unsqueeze(2).expand(-1, -1, N_COND_TOKENS, -1)
        t_pos = t_pos.reshape(B, T * N_COND_TOKENS, D_MODEL)

        cond_tokens = cond_tokens + t_pos

        temporal_pad = ~time_mask
        temporal_pad = temporal_pad.unsqueeze(-1).expand(-1, -1, N_COND_TOKENS)
        temporal_pad = temporal_pad.reshape(B, T * N_COND_TOKENS)

        cond = self.temporal_transformer(cond_tokens, src_key_padding_mask=temporal_pad)

        cond = cond.reshape(B, T, N_COND_TOKENS, D_MODEL)
        ch_id_out = cond[:, :, 0, :]
        ch_id_out_flat = ch_id_out.reshape(B * T, 1, D_MODEL)

        # ── Spatial Decoder Memory ──
        spatial_memory = torch.cat([ch_id_out_flat, telop_memory], dim=1)  # (BT, 1+K_MEMORY, 256)

        # memory padding mask
        any_active = telop_mask_flat.any(dim=1)  # (BT,)
        telop_mem_pad = ~any_active.unsqueeze(1).expand(-1, K_MEMORY)  # (BT, K_MEMORY)
        ch_pad = torch.zeros(B * T, 1, dtype=torch.bool, device=device)
        memory_pad_mask = torch.cat([ch_pad, telop_mem_pad], dim=1)  # (BT, 1+K_MEMORY)

        # ── Spatial Queries with (x, y, t) ──
        xy_emb = self._get_xy_emb(device)

        t_emb = self.time_proj(t_input)
        t_emb_flat = t_emb.reshape(B * T, 1, D_MODEL)

        queries = self.spatial_queries.unsqueeze(0) + xy_emb.unsqueeze(0) + t_emb_flat

        spatial_out = self.spatial_decoder(
            queries, spatial_memory,
            memory_key_padding_mask=memory_pad_mask,
        )

        spatial_2d = spatial_out.reshape(B * T, SPATIAL_H, SPATIAL_W, D_MODEL)
        spatial_2d = spatial_2d.permute(0, 3, 1, 2)
        masks = self.upsample(spatial_2d)
        masks = masks.reshape(B, T, GRID_H, GRID_W)

        return masks


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = FrameMaskModel(n_channels=len(channels)).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"🖥️  Device: {device}")
print(f"📐 파라미터: {n_params:,}")

🖥️  Device: cuda
📐 파라미터: 6,679,969


In [ ]:
# %% 셀 4: 학습
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 50
LR = 1e-4
SAVE_DIR = "./model/8_layout_frame_mask_detr"
os.makedirs(SAVE_DIR, exist_ok=True)


def compute_loss(pred, gt, time_mask):
    mask_3d = time_mask.unsqueeze(-1).unsqueeze(-1).expand_as(pred)
    pred_valid = pred[mask_3d]
    gt_valid = gt[mask_3d]
    pos_weight = torch.tensor([POS_WEIGHT], device=pred.device)
    return F.binary_cross_entropy_with_logits(pred_valid, gt_valid, pos_weight=pos_weight)


@torch.no_grad()
def compute_metrics(pred, gt, time_mask, threshold=0.5):
    mask_3d = time_mask.unsqueeze(-1).unsqueeze(-1).expand_as(pred)
    pred_prob = torch.sigmoid(pred[mask_3d])
    gt_valid = gt[mask_3d]

    pred_bin = (pred_prob > threshold).float()
    tp = (pred_bin * gt_valid).sum().item()
    fp = (pred_bin * (1 - gt_valid)).sum().item()
    fn = ((1 - pred_bin) * gt_valid).sum().item()

    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)

    best_f1 = f1
    best_th = threshold
    for th in [0.3, 0.4, 0.6, 0.7]:
        pb = (pred_prob > th).float()
        t = (pb * gt_valid).sum().item()
        f = (pb * (1 - gt_valid)).sum().item()
        n = ((1 - pb) * gt_valid).sum().item()
        p = t / max(t + f, 1)
        r = t / max(t + n, 1)
        f_ = 2 * p * r / max(p + r, 1e-8)
        if f_ > best_f1:
            best_f1 = f_
            best_th = th

    n_total = pred_prob.numel()
    if n_total > 100000:
        idx = torch.randperm(n_total, device=pred_prob.device)[:100000]
        p_sample = pred_prob[idx]
        g_sample = gt_valid[idx]
    else:
        p_sample = pred_prob
        g_sample = gt_valid

    sorted_idx = torch.argsort(p_sample, descending=True)
    g_sorted = g_sample[sorted_idx]
    tp_cum = g_sorted.cumsum(dim=0)
    precision_curve = tp_cum / torch.arange(1, len(g_sorted) + 1, device=g_sorted.device).float()
    ap = (precision_curve * g_sorted).sum().item() / max(g_sorted.sum().item(), 1)

    return {"precision": precision, "recall": recall, "f1": f1,
            "best_f1": best_f1, "best_th": best_th, "ap": ap}


optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
best_eval_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss_sum, train_batches = 0, 0
    for batch in tqdm(train_loader, desc=f"[{epoch}/{EPOCHS}] train", leave=False):
        channel_id = batch["channel_id"].to(device)
        time_feats = batch["time_feats"].to(device)
        telop_feats = batch["telop_feats"].to(device)
        telop_mask = batch["telop_mask"].to(device)
        gt_masks = batch["gt_masks"].to(device)
        time_mask = batch["time_mask"].to(device)

        with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
            pred = model(channel_id, time_feats, telop_feats, telop_mask, time_mask)
            loss = compute_loss(pred, gt_masks, time_mask)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        train_loss_sum += loss.item()
        train_batches += 1

    model.eval()
    eval_loss_sum, eval_batches = 0, 0
    all_metrics = {k: [] for k in ["precision", "recall", "f1", "best_f1", "best_th", "ap"]}

    with torch.no_grad():
        for batch in eval_loader:
            channel_id = batch["channel_id"].to(device)
            time_feats = batch["time_feats"].to(device)
            telop_feats = batch["telop_feats"].to(device)
            telop_mask = batch["telop_mask"].to(device)
            gt_masks = batch["gt_masks"].to(device)
            time_mask = batch["time_mask"].to(device)

            with torch.amp.autocast(device_type='cuda', dtype=torch.bfloat16):
                pred = model(channel_id, time_feats, telop_feats, telop_mask, time_mask)
                loss = compute_loss(pred, gt_masks, time_mask)
            metrics = compute_metrics(pred, gt_masks, time_mask)

            eval_loss_sum += loss.item()
            eval_batches += 1
            for k, v in metrics.items():
                all_metrics[k].append(v)

    scheduler.step()

    train_loss = train_loss_sum / max(train_batches, 1)
    eval_loss = eval_loss_sum / max(eval_batches, 1)
    avg_m = {k: np.mean(v) for k, v in all_metrics.items()}
    lr_now = optimizer.param_groups[0]["lr"]

    print(
        f"[{epoch:>3}/{EPOCHS}]  "
        f"train={train_loss:.4f}  eval={eval_loss:.4f}  "
        f"P={avg_m['precision']:.3f}  R={avg_m['recall']:.3f}  F1={avg_m['f1']:.3f}  "
        f"bestF1={avg_m['best_f1']:.3f}@{avg_m['best_th']:.1f}  "
        f"AP={avg_m['ap']:.3f}  "
        f"lr={lr_now:.2e}"
    )

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "eval_loss": eval_loss,
        "eval_metrics": avg_m,
        "channel2id": channel2id,
    }, os.path.join(SAVE_DIR, f"epoch_{epoch:03d}.pt"))

    if eval_loss < best_eval_loss:
        best_eval_loss = eval_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "eval_loss": eval_loss,
            "eval_metrics": avg_m,
            "channel2id": channel2id,
        }, os.path.join(SAVE_DIR, "best.pt"))
        print(f"   💾 best 갱신 (eval_loss={eval_loss:.4f})")

print(f"\n✅ 완료. Best eval_loss={best_eval_loss:.4f}")

[1/50] train:   7%|▋         | 2337/31540 [09:32<3:31:18,  2.30it/s]